In [2]:
# [환경 설정]
# DuckDB가 대용량 집계 시 사용할 메모리와 임시 저장공간 설정

from pathlib import Path
import duckdb

temp_dir = Path(r"D:\duckdb_temp")
temp_dir.mkdir(exist_ok=True)

duckdb.sql("SET temp_directory = 'D:/duckdb_temp'")
duckdb.sql("SET preserve_insertion_order = false")
duckdb.sql("SET threads = 2")
duckdb.sql("SET memory_limit = '4GB'")

In [4]:
# 세션 행동 분석을 위한 기본 설정

import duckdb
from pathlib import Path

parquet_path = r"../data/processed/20*.parquet"

analysis_start_date = "2019-12-01"
analysis_end_date = "2020-05-01"

anomaly_dates = [
    "2020-01-01",
    "2020-01-02",
    "2020-01-03",
    "2020-02-27",
    "2020-04-20",
    "2020-04-21"
]

anomaly_dates_sql = ", ".join(f"'{date}'" for date in anomaly_dates)

In [8]:
# [분석용 코드]
# 세션별 기본 행동량 생성
# unique_products는 메모리 부담 때문에 다음 단계에서 별도로 계산

session_base_parquet = r"../data/processed/session_behavior_base.parquet"

Path(session_base_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            user_session,

            MIN(event_time) AS session_start,
            MAX(event_time) AS session_end,

            COUNT(*) AS total_events,

            SUM(
                CASE WHEN event_type = 'view'
                THEN 1 ELSE 0 END
            ) AS view_count,

            SUM(
                CASE WHEN event_type = 'cart'
                THEN 1 ELSE 0 END
            ) AS cart_count,

            SUM(
                CASE WHEN event_type = 'purchase'
                THEN 1 ELSE 0 END
            ) AS purchase_count

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})

        GROUP BY user_session
    )
    TO '{session_base_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
# user_session 하나가 실제로 얼마나 오래 지속되는지 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS session_count,

        ROUND(
            MEDIAN(
                DATE_DIFF('second', session_start, session_end) / 60.0
            ),
            2
        ) AS median_session_minutes,

        ROUND(
            QUANTILE_CONT(
                DATE_DIFF('second', session_start, session_end) / 60.0,
                0.99
            ),
            2
        ) AS p99_session_minutes,

        ROUND(
            MAX(
                DATE_DIFF('second', session_start, session_end) / 60.0
            ),
            2
        ) AS max_session_minutes,

        MAX(total_events) AS max_total_events

    FROM read_parquet('{session_base_parquet}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬────────────────────────┬─────────────────────┬─────────────────────┬──────────────────┐
│ session_count │ median_session_minutes │ p99_session_minutes │ max_session_minutes │ max_total_events │
│     int64     │         double         │       double        │       double        │      int64       │
├───────────────┼────────────────────────┼─────────────────────┼─────────────────────┼──────────────────┤
│      64698911 │                   0.57 │              647.05 │           218611.48 │            34570 │
└───────────────┴────────────────────────┴─────────────────────┴─────────────────────┴──────────────────┘



In [12]:
# 지속시간이 가장 긴 user_session의 실제 기간과 이벤트 수 확인

duckdb.sql(f"""
    SELECT
        user_session,
        session_start,
        session_end,

        ROUND(
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0,
            2
        ) AS session_minutes,

        total_events,
        view_count,
        cart_count,
        purchase_count

    FROM read_parquet('{session_base_parquet}')

    ORDER BY session_minutes DESC

    LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬─────────────────────┬─────────────────────┬─────────────────┬──────────────┬────────────┬────────────┬────────────────┐
│             user_session             │    session_start    │     session_end     │ session_minutes │ total_events │ view_count │ cart_count │ purchase_count │
│               varchar                │      timestamp      │      timestamp      │     double      │    int64     │   double   │   double   │     double     │
├──────────────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────┼──────────────┼────────────┼────────────┼────────────────┤
│ 69f75110-556e-4a46-88eb-b62d347df229 │ 2019-12-01 00:48:48 │ 2020-04-30 20:20:17 │       218611.48 │          549 │      548.0 │        1.0 │            0.0 │
│ 14f59255-8bbd-4ce9-be82-d6f307c5f12d │ 2019-12-01 06:23:33 │ 2020-04-30 09:21:45 │        217618.2 │            9 │        9.0 │        0.0 │            0.0 │
│ 46959156-d61c-4f8e-a9c7-02764f0d

In [16]:
# 장시간 지속되는 user_session의 규모 확인

duckdb.sql(f"""
    WITH session_duration AS (
        SELECT
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0 AS session_minutes

        FROM read_parquet('{session_base_parquet}')
    )

    SELECT
        COUNT(*) AS total_sessions,

        SUM(CASE WHEN session_minutes > 60
                 THEN 1 ELSE 0 END) AS over_1_hour,

        SUM(CASE WHEN session_minutes > 1440
                 THEN 1 ELSE 0 END) AS over_1_day,

        SUM(CASE WHEN session_minutes > 10080
                 THEN 1 ELSE 0 END) AS over_7_days,

        SUM(CASE WHEN session_minutes > 43200
                 THEN 1 ELSE 0 END) AS over_30_days

    FROM session_duration
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────┬────────────┬─────────────┬──────────────┐
│ total_sessions │ over_1_hour │ over_1_day │ over_7_days │ over_30_days │
│     int64      │   int128    │   int128   │   int128    │    int128    │
├────────────────┼─────────────┼────────────┼─────────────┼──────────────┤
│       64698911 │     1506682 │     540778 │      348403 │       188965 │
└────────────────┴─────────────┴────────────┴─────────────┴──────────────┘



In [18]:
# 하나의 user_session이 여러 user_id에 연결되는지 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS multi_user_session_count

    FROM (
        SELECT
            user_session

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})

        GROUP BY user_session

        HAVING COUNT(DISTINCT user_id) > 1
    )
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┐
│ multi_user_session_count │
│          int64           │
├──────────────────────────┤
│                     5872 │
└──────────────────────────┘



In [22]:
# [검증용 코드]
# 장기 user_session 내부에서 이벤트가 실제로 어떤 간격으로 발생하는지 확인

duckdb.sql(f"""
    WITH long_sessions AS (
        SELECT
            user_session

        FROM read_parquet('{session_base_parquet}')

        ORDER BY
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) DESC

        LIMIT 20
    ),

    events AS (
        SELECT
            r.user_id,
            r.user_session,
            r.event_time,
            r.event_type,
            r.product_id,

            LAG(r.event_time) OVER (
                PARTITION BY r.user_id, r.user_session
                ORDER BY r.event_time
            ) AS previous_event_time

        FROM read_parquet('{parquet_path}') r

        JOIN long_sessions l
            ON r.user_session = l.user_session

        WHERE r.event_time >= '{analysis_start_date}'
          AND r.event_time < '{analysis_end_date}'
    )

    SELECT
        user_id,
        user_session,
        event_time,
        event_type,
        product_id,
        previous_event_time,

        ROUND(
            DATE_DIFF(
                'second',
                previous_event_time,
                event_time
            ) / 60.0,
            2
        ) AS gap_minutes

    FROM events

    ORDER BY
        user_session,
        event_time
""").show(max_rows=200)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬──────────────────────────────────────┬─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────┐
│  user_id  │             user_session             │     event_time      │ event_type │ product_id │ previous_event_time │ gap_minutes │
│   int64   │               varchar                │      timestamp      │  varchar   │   int64    │      timestamp      │   double    │
├───────────┼──────────────────────────────────────┼─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────┤
│ 580484215 │ 0521326c-3000-4e19-9cff-672524050d4d │ 2019-12-01 19:46:48 │ view       │    3601522 │ NULL                │        NULL │
│ 580484215 │ 0521326c-3000-4e19-9cff-672524050d4d │ 2019-12-01 19:46:58 │ cart       │    3601522 │ 2019-12-01 19:46:48 │        0.17 │
│ 580484215 │ 0521326c-3000-4e19-9cff-672524050d4d │ 2019-12-01 19:47:13 │ view       │    3601522 │ 2019-12-01 19:46:58 │        0.25 │
│ 580484215 │ 0521326c-3000-4e19-9cff-672

In [24]:
# 같은 user_id + user_session 안에서 이벤트 간 공백 분포 확인

duckdb.sql(f"""
    WITH ordered_events AS (
        SELECT
            user_id,
            user_session,
            event_time,

            LAG(event_time) OVER (
                PARTITION BY user_id, user_session
                ORDER BY event_time
            ) AS previous_event_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})
    ),

    gaps AS (
        SELECT
            DATE_DIFF(
                'second',
                previous_event_time,
                event_time
            ) / 60.0 AS gap_minutes

        FROM ordered_events

        WHERE previous_event_time IS NOT NULL
    )

    SELECT
        COUNT(*) AS gap_count,

        ROUND(MEDIAN(gap_minutes), 2) AS median_gap,

        ROUND(
            QUANTILE_CONT(gap_minutes, 0.90),
            2
        ) AS p90_gap,

        ROUND(
            QUANTILE_CONT(gap_minutes, 0.95),
            2
        ) AS p95_gap,

        ROUND(
            QUANTILE_CONT(gap_minutes, 0.99),
            2
        ) AS p99_gap,

        ROUND(
            QUANTILE_CONT(gap_minutes, 0.999),
            2
        ) AS p999_gap,

        ROUND(MAX(gap_minutes), 2) AS max_gap

    FROM gaps
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬─────────┬─────────┬─────────┬──────────┬───────────┐
│ gap_count │ median_gap │ p90_gap │ p95_gap │ p99_gap │ p999_gap │  max_gap  │
│   int64   │   double   │ double  │ double  │ double  │  double  │  double   │
├───────────┼────────────┼─────────┼─────────┼─────────┼──────────┼───────────┤
│ 226898398 │       0.53 │    2.32 │    4.03 │    32.2 │ 27336.16 │ 217614.88 │
└───────────┴────────────┴─────────┴─────────┴─────────┴──────────┴───────────┘



In [25]:
# 이벤트 간 공백을 구간별로 나누어 분포 확인

duckdb.sql(f"""
    WITH ordered_events AS (
        SELECT
            user_id,
            user_session,
            event_time,

            LAG(event_time) OVER (
                PARTITION BY user_id, user_session
                ORDER BY event_time
            ) AS previous_event_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})
    ),

    gaps AS (
        SELECT
            DATE_DIFF(
                'second',
                previous_event_time,
                event_time
            ) / 60.0 AS gap_minutes

        FROM ordered_events

        WHERE previous_event_time IS NOT NULL
    )

    SELECT
        CASE
            WHEN gap_minutes < 5 THEN '5분 미만'
            WHEN gap_minutes < 10 THEN '5~10분'
            WHEN gap_minutes < 30 THEN '10~30분'
            WHEN gap_minutes < 60 THEN '30~60분'
            WHEN gap_minutes < 180 THEN '1~3시간'
            WHEN gap_minutes < 1440 THEN '3~24시간'
            ELSE '24시간 이상'
        END AS gap_group,

        COUNT(*) AS gap_count,

        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            3
        ) AS gap_share

    FROM gaps

    GROUP BY gap_group

    ORDER BY
        CASE gap_group
            WHEN '5분 미만' THEN 1
            WHEN '5~10분' THEN 2
            WHEN '10~30분' THEN 3
            WHEN '30~60분' THEN 4
            WHEN '1~3시간' THEN 5
            WHEN '3~24시간' THEN 6
            ELSE 7
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬───────────┬───────────┐
│  gap_group  │ gap_count │ gap_share │
│   varchar   │   int64   │  double   │
├─────────────┼───────────┼───────────┤
│ 5분 미만    │ 218079695 │    96.113 │
│ 5~10분      │   4334985 │     1.911 │
│ 10~30분     │   2135519 │     0.941 │
│ 30~60분     │    637929 │     0.281 │
│ 1~3시간     │    543307 │     0.239 │
│ 3~24시간    │    485170 │     0.214 │
│ 24시간 이상 │    681793 │       0.3 │
└─────────────┴───────────┴───────────┘



In [45]:
# 30분 이상 활동 공백을 기준으로 분석 세션 재구성
# 동일한 event_time의 여러 이벤트가 서로 다른 세션으로 분리되지 않도록 처리

sessionized_events_parquet = r"../data/processed/sessionized_events.parquet"

Path(sessionized_events_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        WITH ordered_events AS (
            SELECT
                user_id,
                user_session,
                event_time,
                event_type,
                product_id,

                LAG(event_time) OVER (
                    PARTITION BY user_id, user_session
                    ORDER BY event_time
                ) AS previous_event_time

            FROM read_parquet('{parquet_path}')

            WHERE user_session IS NOT NULL
              AND event_time >= '{analysis_start_date}'
              AND event_time < '{analysis_end_date}'
              AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})
        ),

        session_flags AS (
            SELECT
                *,

                CASE
                    WHEN previous_event_time IS NULL
                      OR DATE_DIFF(
                            'second',
                            previous_event_time,
                            event_time
                         ) >= 1800
                    THEN 1
                    ELSE 0
                END AS new_session_flag

            FROM ordered_events
        )

        SELECT
            user_id,
            user_session,
            event_time,
            event_type,
            product_id,

            SUM(new_session_flag) OVER (
                PARTITION BY user_id, user_session
                ORDER BY event_time
                RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS analysis_session_number

        FROM session_flags
    )
    TO '{sessionized_events_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [46]:
# 원본 session ID가 몇 개의 분석 세션으로 분리되었는지 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS original_user_sessions,

        SUM(analysis_session_count) AS analysis_sessions,

        MAX(analysis_session_count) AS max_split_count

    FROM (
        SELECT
            user_id,
            user_session,
            MAX(analysis_session_number) AS analysis_session_count

        FROM read_parquet('{sessionized_events_parquet}')

        GROUP BY
            user_id,
            user_session
    )
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┬───────────────────┬─────────────────┐
│ original_user_sessions │ analysis_sessions │ max_split_count │
│         int64          │      double       │     double      │
├────────────────────────┼───────────────────┼─────────────────┤
│               64704846 │        67053045.0 │           373.0 │
└────────────────────────┴───────────────────┴─────────────────┘



In [52]:
# 재구성한 분석 세션별 View / Cart / Purchase 행동량 생성
# 메모리 부담이 큰 COUNT(DISTINCT product_id)는 우선 제외

analysis_session_parquet = r"../data/processed/analysis_session_behavior.parquet"

Path(analysis_session_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            user_id,
            user_session,
            analysis_session_number,

            MIN(event_time) AS session_start,
            MAX(event_time) AS session_end,

            COUNT(*) AS total_events,

            SUM(
                CASE WHEN event_type = 'view'
                THEN 1 ELSE 0 END
            ) AS view_count,

            SUM(
                CASE WHEN event_type = 'cart'
                THEN 1 ELSE 0 END
            ) AS cart_count,

            SUM(
                CASE WHEN event_type = 'purchase'
                THEN 1 ELSE 0 END
            ) AS purchase_count

        FROM read_parquet('{sessionized_events_parquet}')

        GROUP BY
            user_id,
            user_session,
            analysis_session_number
    )
    TO '{analysis_session_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [54]:
# 재구성한 분석 세션의 규모와 지속시간 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS session_count,

        ROUND(
            MEDIAN(
                DATE_DIFF(
                    'second',
                    session_start,
                    session_end
                ) / 60.0
            ),
            2
        ) AS median_session_minutes,

        ROUND(
            QUANTILE_CONT(
                DATE_DIFF(
                    'second',
                    session_start,
                    session_end
                ) / 60.0,
                0.99
            ),
            2
        ) AS p99_session_minutes,

        ROUND(
            MAX(
                DATE_DIFF(
                    'second',
                    session_start,
                    session_end
                ) / 60.0
            ),
            2
        ) AS max_session_minutes,

        MAX(total_events) AS max_total_events

    FROM read_parquet('{analysis_session_parquet}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬────────────────────────┬─────────────────────┬─────────────────────┬──────────────────┐
│ session_count │ median_session_minutes │ p99_session_minutes │ max_session_minutes │ max_total_events │
│     int64     │         double         │       double        │       double        │      int64       │
├───────────────┼────────────────────────┼─────────────────────┼─────────────────────┼──────────────────┤
│      67053045 │                   0.45 │               37.28 │             8693.22 │            32639 │
└───────────────┴────────────────────────┴─────────────────────┴─────────────────────┴──────────────────┘



In [55]:
# 가장 긴 분석 세션의 지속시간과 행동량 확인

duckdb.sql(f"""
    SELECT
        user_id,
        user_session,
        analysis_session_number,

        session_start,
        session_end,

        ROUND(
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0,
            2
        ) AS session_minutes,

        total_events,
        view_count,
        cart_count,
        purchase_count

    FROM read_parquet('{analysis_session_parquet}')

    ORDER BY session_minutes DESC

    LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬──────────────────────────────────────┬─────────────────────────┬─────────────────────┬─────────────────────┬─────────────────┬──────────────┬────────────┬────────────┬────────────────┐
│  user_id  │             user_session             │ analysis_session_number │    session_start    │     session_end     │ session_minutes │ total_events │ view_count │ cart_count │ purchase_count │
│   int64   │               varchar                │         double          │      timestamp      │      timestamp      │     double      │    int64     │   double   │   double   │     double     │
├───────────┼──────────────────────────────────────┼─────────────────────────┼─────────────────────┼─────────────────────┼─────────────────┼──────────────┼────────────┼────────────┼────────────────┤
│ 550985043 │ 8a9edb0e-9d66-bb0a-064f-47c75b51c805 │                     2.0 │ 2020-03-29 19:06:26 │ 2020-04-04 19:59:39 │         8693.22 │         1979 │     1979.0 │        0.0 │            0.0 │
│ 550

In [58]:
# 가장 긴 분석 세션 내부의 이벤트와 이벤트 간 공백 직접 확인

target_user_id = 556670855
target_user_session = "34c981e3-d914-46b8-8d54-063c74e2b913"
target_analysis_session = 1

duckdb.sql(f"""
    WITH target_events AS (
        SELECT
            user_id,
            user_session,
            analysis_session_number,
            event_time,
            event_type,
            product_id,

            LAG(event_time) OVER (
                ORDER BY event_time
            ) AS previous_event_time

        FROM read_parquet('{sessionized_events_parquet}')

        WHERE user_id = {target_user_id}
          AND user_session = '{target_user_session}'
          AND analysis_session_number = {target_analysis_session}
    )

    SELECT
        event_time,
        event_type,
        product_id,
        previous_event_time,

        ROUND(
            DATE_DIFF(
                'second',
                previous_event_time,
                event_time
            ) / 60.0,
            2
        ) AS gap_minutes

    FROM target_events

    ORDER BY event_time
""").show(max_rows=100)

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────┐
│     event_time      │ event_type │ product_id │ previous_event_time │ gap_minutes │
│      timestamp      │  varchar   │   int64    │      timestamp      │   double    │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────┤
│ 2019-12-03 10:57:44 │ view       │  100016733 │ NULL                │        NULL │
│ 2019-12-03 10:58:09 │ view       │  100016733 │ 2019-12-03 10:57:44 │        0.42 │
│ 2019-12-03 10:59:35 │ view       │  100017097 │ 2019-12-03 10:58:09 │        1.43 │
│ 2019-12-03 11:00:49 │ view       │  100016733 │ 2019-12-03 10:59:35 │        1.23 │
│ 2019-12-03 11:01:01 │ cart       │  100016733 │ 2019-12-03 11:00:49 │         0.2 │
│ 2019-12-03 11:01:50 │ view       │  100016733 │ 2019-12-03 11:01:01 │        0.82 │
│ 2019-12-03 11:01:50 │ cart       │  100016733 │ 2019-12-03 11:01:50 │         0.0 │
└─────────────────────┴────────────┴────────────┴─────

In [62]:
# 재구성 후에도 남아 있는 장시간 분석 세션의 규모 확인

duckdb.sql(f"""
    WITH session_duration AS (
        SELECT
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0 AS session_minutes

        FROM read_parquet('{analysis_session_parquet}')
    )

    SELECT
        COUNT(*) AS total_sessions,

        SUM(CASE WHEN session_minutes > 60
                 THEN 1 ELSE 0 END) AS over_1_hour,

        SUM(CASE WHEN session_minutes > 180
                 THEN 1 ELSE 0 END) AS over_3_hours,

        SUM(CASE WHEN session_minutes > 720
                 THEN 1 ELSE 0 END) AS over_12_hours,

        SUM(CASE WHEN session_minutes > 1440
                 THEN 1 ELSE 0 END) AS over_1_day

    FROM session_duration
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────┬──────────────┬───────────────┬────────────┐
│ total_sessions │ over_1_hour │ over_3_hours │ over_12_hours │ over_1_day │
│     int64      │   int128    │    int128    │    int128     │   int128   │
├────────────────┼─────────────┼──────────────┼───────────────┼────────────┤
│       67053045 │      166227 │         1835 │            18 │          7 │
└────────────────┴─────────────┴──────────────┴───────────────┴────────────┘



In [65]:
#본 분석 단계

In [67]:
# 분석 세션을 구매 세션 / 비구매 세션으로 분류

duckdb.sql(f"""
    SELECT
        CASE
            WHEN purchase_count > 0 THEN '구매 세션'
            ELSE '비구매 세션'
        END AS session_type,

        COUNT(*) AS session_count

    FROM read_parquet('{analysis_session_parquet}')

    GROUP BY session_type
    ORDER BY session_type
""").show()

┌──────────────┬───────────────┐
│ session_type │ session_count │
│   varchar    │     int64     │
├──────────────┼───────────────┤
│ 구매 세션    │       4017607 │
│ 비구매 세션  │      63035438 │
└──────────────┴───────────────┘



In [69]:
# 구매 세션과 비구매 세션의 행동량 및 세션 길이 비교

duckdb.sql(f"""
    SELECT
        CASE
            WHEN purchase_count > 0 THEN '구매 세션'
            ELSE '비구매 세션'
        END AS session_type,

        COUNT(*) AS session_count,

        ROUND(AVG(total_events), 2) AS avg_total_events,
        ROUND(MEDIAN(total_events), 2) AS median_total_events,

        ROUND(AVG(view_count), 2) AS avg_view_count,
        ROUND(MEDIAN(view_count), 2) AS median_view_count,

        ROUND(AVG(cart_count), 2) AS avg_cart_count,
        ROUND(MEDIAN(cart_count), 2) AS median_cart_count,

        ROUND(
            AVG(
                DATE_DIFF(
                    'second',
                    session_start,
                    session_end
                ) / 60.0
            ),
            2
        ) AS avg_session_minutes,

        ROUND(
            MEDIAN(
                DATE_DIFF(
                    'second',
                    session_start,
                    session_end
                ) / 60.0
            ),
            2
        ) AS median_session_minutes

    FROM read_parquet('{analysis_session_parquet}')

    GROUP BY session_type
    ORDER BY session_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬───────────────┬──────────────────┬─────────────────────┬────────────────┬───────────────────┬────────────────┬───────────────────┬─────────────────────┬────────────────────────┐
│ session_type │ session_count │ avg_total_events │ median_total_events │ avg_view_count │ median_view_count │ avg_cart_count │ median_cart_count │ avg_session_minutes │ median_session_minutes │
│   varchar    │     int64     │      double      │       double        │     double     │      double       │     double     │      double       │       double        │         double         │
├──────────────┼───────────────┼──────────────────┼─────────────────────┼────────────────┼───────────────────┼────────────────┼───────────────────┼─────────────────────┼────────────────────────┤
│ 구매 세션    │       4017607 │             8.89 │                 6.0 │           5.75 │               3.0 │           1.86 │               1.0 │                7.71 │                   4.12 │
│ 비구매 세션  │      63035438 │  

In [75]:
# 세션의 View 횟수 구간별 구매 세션 비율 확인

duckdb.sql(f"""
    WITH session_group AS (
        SELECT
            CASE
                WHEN view_count = 0 THEN '0회'
                WHEN view_count = 1 THEN '1회'
                WHEN view_count BETWEEN 2 AND 3 THEN '2~3회'
                WHEN view_count BETWEEN 4 AND 5 THEN '4~5회'
                WHEN view_count BETWEEN 6 AND 10 THEN '6~10회'
                ELSE '11회 이상'
            END AS view_group,

            CASE
                WHEN purchase_count > 0 THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{analysis_session_parquet}')
    )

    SELECT
        view_group,

        COUNT(*) AS session_count,
        SUM(purchased) AS purchase_session_count,

        ROUND(
            SUM(purchased) * 100.0
            / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_group

    GROUP BY view_group

    ORDER BY
        CASE view_group
            WHEN '0회' THEN 1
            WHEN '1회' THEN 2
            WHEN '2~3회' THEN 3
            WHEN '4~5회' THEN 4
            WHEN '6~10회' THEN 5
            ELSE 6
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬────────────────────────┬───────────────────────┐
│ view_group │ session_count │ purchase_session_count │ purchase_session_rate │
│  varchar   │     int64     │         int128         │        double         │
├────────────┼───────────────┼────────────────────────┼───────────────────────┤
│ 0회        │        499401 │                  51820 │                 10.38 │
│ 1회        │      30365001 │                 717303 │                  2.36 │
│ 2~3회      │      16515712 │                1464531 │                  8.87 │
│ 4~5회      │       6851405 │                 634864 │                  9.27 │
│ 6~10회     │       7089623 │                 621359 │                  8.76 │
│ 11회 이상  │       5731903 │                 527730 │                  9.21 │
└────────────┴───────────────┴────────────────────────┴───────────────────────┘



In [77]:
# View가 0회인 세션의 Cart / Purchase 구성 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS session_count,

        SUM(
            CASE WHEN cart_count > 0
            THEN 1 ELSE 0 END
        ) AS cart_session_count,

        SUM(
            CASE WHEN purchase_count > 0
            THEN 1 ELSE 0 END
        ) AS purchase_session_count,

        SUM(
            CASE
                WHEN cart_count = 0
                 AND purchase_count > 0
                THEN 1 ELSE 0
            END
        ) AS purchase_without_cart_count

    FROM read_parquet('{analysis_session_parquet}')

    WHERE view_count = 0
""").show()

┌───────────────┬────────────────────┬────────────────────────┬─────────────────────────────┐
│ session_count │ cart_session_count │ purchase_session_count │ purchase_without_cart_count │
│     int64     │       int128       │         int128         │           int128            │
├───────────────┼────────────────────┼────────────────────────┼─────────────────────────────┤
│        499401 │             456257 │                  51820 │                       43144 │
└───────────────┴────────────────────┴────────────────────────┴─────────────────────────────┘



In [79]:
# 원본 user_id + user_session 기준으로 View 없이 Purchase가 발생한 세션 확인

duckdb.sql(f"""
    WITH original_sessions AS (
        SELECT
            user_id,
            user_session,

            SUM(
                CASE WHEN event_type = 'view'
                THEN 1 ELSE 0 END
            ) AS view_count,

            SUM(
                CASE WHEN event_type = 'purchase'
                THEN 1 ELSE 0 END
            ) AS purchase_count

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})

        GROUP BY
            user_id,
            user_session
    )

    SELECT
        COUNT(*) AS original_session_count,

        SUM(
            CASE
                WHEN view_count = 0
                 AND purchase_count > 0
                THEN 1 ELSE 0
            END
        ) AS purchase_without_view_count

    FROM original_sessions
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┬─────────────────────────────┐
│ original_session_count │ purchase_without_view_count │
│         int64          │           int128            │
├────────────────────────┼─────────────────────────────┤
│               64704846 │                       45231 │
└────────────────────────┴─────────────────────────────┘



In [80]:
# View=0인 구매 분석 세션 중 원본 user_session에는 View가 존재했는지 확인

duckdb.sql(f"""
    WITH purchase_without_view AS (
        SELECT
            user_id,
            user_session,
            analysis_session_number

        FROM read_parquet('{analysis_session_parquet}')

        WHERE view_count = 0
          AND purchase_count > 0
    ),

    original_view AS (
        SELECT DISTINCT
            user_id,
            user_session

        FROM read_parquet('{sessionized_events_parquet}')

        WHERE event_type = 'view'
    )

    SELECT
        COUNT(*) AS purchase_without_view_sessions,

        SUM(
            CASE
                WHEN o.user_session IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS original_session_had_view,

        SUM(
            CASE
                WHEN o.user_session IS NULL
                THEN 1 ELSE 0
            END
        ) AS original_session_also_no_view

    FROM purchase_without_view p

    LEFT JOIN original_view o
        ON p.user_id = o.user_id
       AND p.user_session = o.user_session
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────┬───────────────────────────┬───────────────────────────────┐
│ purchase_without_view_sessions │ original_session_had_view │ original_session_also_no_view │
│             int64              │          int128           │            int128             │
├────────────────────────────────┼───────────────────────────┼───────────────────────────────┤
│                          51820 │                      6508 │                         45312 │
└────────────────────────────────┴───────────────────────────┴───────────────────────────────┘



In [84]:
# 세션 길이 구간별 구매 세션 비율 확인

duckdb.sql(f"""
    WITH session_duration AS (
        SELECT
            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0 AS session_minutes,

            CASE
                WHEN purchase_count > 0 THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{analysis_session_parquet}')
    ),

    session_group AS (
        SELECT
            CASE
                WHEN session_minutes < 1 THEN '1분 미만'
                WHEN session_minutes < 3 THEN '1~3분'
                WHEN session_minutes < 5 THEN '3~5분'
                WHEN session_minutes < 10 THEN '5~10분'
                WHEN session_minutes < 30 THEN '10~30분'
                ELSE '30분 이상'
            END AS duration_group,

            purchased

        FROM session_duration
    )

    SELECT
        duration_group,
        COUNT(*) AS session_count,
        SUM(purchased) AS purchase_session_count,

        ROUND(
            SUM(purchased) * 100.0
            / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_group

    GROUP BY duration_group

    ORDER BY
        CASE duration_group
            WHEN '1분 미만' THEN 1
            WHEN '1~3분' THEN 2
            WHEN '3~5분' THEN 3
            WHEN '5~10분' THEN 4
            WHEN '10~30분' THEN 5
            ELSE 6
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬───────────────┬────────────────────────┬───────────────────────┐
│ duration_group │ session_count │ purchase_session_count │ purchase_session_rate │
│    varchar     │     int64     │         int128         │        double         │
├────────────────┼───────────────┼────────────────────────┼───────────────────────┤
│ 1분 미만       │      38343583 │                 343104 │                  0.89 │
│ 1~3분          │      10299429 │                1200404 │                 11.66 │
│ 3~5분          │       5283095 │                 737844 │                 13.97 │
│ 5~10분         │       6283094 │                 851041 │                 13.54 │
│ 10~30분        │       5696796 │                 721858 │                 12.67 │
│ 30분 이상      │       1147048 │                 163356 │                 14.24 │
└────────────────┴───────────────┴────────────────────────┴───────────────────────┘



In [86]:
# 월별 구매 세션 / 비구매 세션의 View 수와 세션 길이 비교

duckdb.sql(f"""
    WITH session_summary AS (
        SELECT
            STRFTIME(session_start, '%Y-%m') AS month,

            CASE
                WHEN purchase_count > 0 THEN '구매 세션'
                ELSE '비구매 세션'
            END AS session_type,

            view_count,

            DATE_DIFF(
                'second',
                session_start,
                session_end
            ) / 60.0 AS session_minutes

        FROM read_parquet('{analysis_session_parquet}')
    )

    SELECT
        month,
        session_type,
        COUNT(*) AS session_count,

        ROUND(AVG(view_count), 2) AS avg_view_count,
        MEDIAN(view_count) AS median_view_count,

        ROUND(AVG(session_minutes), 2) AS avg_session_minutes,
        ROUND(MEDIAN(session_minutes), 2) AS median_session_minutes

    FROM session_summary

    GROUP BY
        month,
        session_type

    ORDER BY
        month,
        session_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┬───────────────┬────────────────┬───────────────────┬─────────────────────┬────────────────────────┐
│  month  │ session_type │ session_count │ avg_view_count │ median_view_count │ avg_session_minutes │ median_session_minutes │
│ varchar │   varchar    │     int64     │     double     │      double       │       double        │         double         │
├─────────┼──────────────┼───────────────┼────────────────┼───────────────────┼─────────────────────┼────────────────────────┤
│ 2019-12 │ 구매 세션    │        972055 │           5.56 │               3.0 │                7.35 │                   4.02 │
│ 2019-12 │ 비구매 세션  │      14951390 │           3.85 │               2.0 │                3.14 │                    0.3 │
│ 2020-01 │ 구매 세션    │        669888 │           5.31 │               3.0 │                6.72 │                   3.68 │
│ 2020-01 │ 비구매 세션  │      12238974 │            3.6 │               1.0 │                2.71 │                    0.0 │
│ 